# 임상 ML 학습·평가·개인 계정 MLflow 기록

실행 설정에서 `training_mode`를 `yaml` 또는 `optuna`로 선택하고 위에서 아래로 실행합니다. 선택한 경로만 학습하며, 완료된 모델과 validation 지표를 각 PC의 개인 계정으로 자동 기록합니다. 기본 `run_final_test=True`이면 같은 노트북 마지막 단계에서 선택한 모델을 test 평가하고 MLflow에 기록합니다.

모델·입력 버전·threshold를 비교 중이면 `run_final_test=False`, 오프라인 실행은 `mlflow_logging=False`를 사용합니다. 학습 전 계정 확인 실패는 중단하고, 학습 후 전송 실패는 로컬 모델을 보존해 재전송할 수 있습니다. Optuna는 완료된 study의 best 한 개만 기록합니다.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Members 패키지를 찾도록 저장소 루트를 Python 검색 경로에 등록합니다.
repo = Path.cwd().resolve()
while not (repo / "Members/sang/model/ML/model_choice.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("sixletters-poc 저장소 내부에서 노트북을 실행하세요")
    repo = repo.parent

sys.path.insert(0, str(repo))
choice_path = repo / "Members/sang/model/ML/model_choice.yaml"

from Members.sang.model.ML.training import train_clinical  # noqa: E402

print("Members import 준비 완료:", repo)

## 입력 변수 버전 선택

`model_version`을 **1, 2, 3, 4 중 하나**로 지정합니다. 기본값 4는 기존 전체 23개 입력을 유지합니다. 모든 알고리즘(XGBoost·CatBoost·Random Forest)과 Optuna에 같은 버전을 적용합니다.

| model_version | 입력 변수 수 | 구성 |
|---|---:|---|
| 1 | 4 | 성별, 나이, LVEF, NYHA |
| 2 | 8 | 버전 1 + eGFR, 허혈성 원인, Albumin, 당뇨 |
| 3 | 21 | 버전 2 + 제공한 심장 구조·기능, 병력, 혈압, Pro-BNP, 심흉비 |
| 4 | 23 | 버전 3 + 흡연량, 음주량 |

정확한 열 목록과 순서는 `ML/feature_versions.py`에 있습니다. `Mitral valve insufficiency `의 끝 공백은 CSV 로더와 동일하게 정리했습니다. 범주형 열도 선택한 입력 변수 안에서만 처리합니다. 아래 출력에서 실제 입력 열을 확인하세요.

환자 split은 모든 버전에서 같습니다. 버전이 다른 저장 모델은 재사용하지 않으며, 저장 결과에 알고리즘 이름과 별도로 `model_version`·`feature_names`를 기록합니다. 버전 비교와 최적값 선택은 validation으로 하고 test는 최종 평가에 사용합니다.

아래 셀에서 학습 방식, 개인 계정 자동 기록, 최종 평가를 함께 설정합니다. `yaml`은 `model_choice.yaml`의 모델 목록을 학습하고, `optuna`는 `model_name` 하나를 탐색합니다. 전체 셀 실행으로 두 경로를 중복 실행하지 않습니다.

In [ ]:
from Members.sang.model.ML.feature_versions import get_feature_columns

model_version = 4  # 1: 4개, 2: 8개, 3: 21개, 4: 23개
training_mode = "optuna"  # "yaml": 고정값 학습, "optuna": best 한 개 탐색
model_name = "XGBoost"  # Optuna: XGBoost / CatBoost / Random Forest
n_trials = 100
reuse_saved = False  # yaml: True이면 동일 입력 버전의 저장 모델 재사용
mlflow_logging = True
run_final_test = True  # 모델·입력 버전·threshold를 비교 중이면 False
retry_mlflow = False  # 전송 실패 후 재시도 셀만 실행할 때 True

if training_mode not in {"yaml", "optuna"}:
    raise ValueError("training_mode는 yaml 또는 optuna를 사용하세요")
if model_name not in {"XGBoost", "CatBoost", "Random Forest"}:
    raise ValueError("지원되는 model_name을 선택하세요")
if n_trials < 1:
    raise ValueError("n_trials는 1 이상이어야 합니다")
run_dirs = []
best_run_dir = None
evaluation_dirs = []
input_columns = get_feature_columns(model_version)
print("model_version:", model_version, "training_mode:", training_mode)
print("입력 변수 수:", len(input_columns))
input_columns

## YAML 고정값 학습

`training_mode="yaml"`일 때만 실행합니다. `model_choice.yaml`의 모델 목록과 각 parameter YAML을 사용합니다. 기본 `reuse_saved=False`는 새 모델을 학습합니다. `True`이면 입력 버전·변수 목록이 일치하는 기존 run을 재사용하며, 해당 run도 개인 계정에 기록할 수 있습니다.

각 완료 run의 모델·전처리·파라미터·validation 지표를 자동 기록합니다. 최종 평가는 마지막 공통 평가 셀에서 이 실행의 `run_dirs`를 직접 사용합니다. XGBoost·CatBoost는 validation AP로 best iteration을 선택하고 Random Forest는 설정한 tree 수로 학습합니다.

In [ ]:
if training_mode == "yaml":
    run_dirs = train_clinical(
        choice_path,
        model_version=model_version,
        reuse_saved=reuse_saved,
        mlflow_logging=mlflow_logging,
    )
    print(run_dirs)

## YAML 학습 결과 확인

각 run에 모델·전처리·설정·validation 예측/지표를 저장합니다. 자동 전송 상태와 링크는 `mlflow_status.json` 또는 `summary.json`에 있습니다. 최종 test 평가는 아래 마지막 셀에서 실행합니다.

In [ ]:
if training_mode == "yaml":
    for run_dir in run_dirs:
        print(run_dir.name)
        display(pd.read_csv(run_dir / "validation_predictions.csv").head())
        print(json.loads((run_dir / "metrics.json").read_text())["validation"])
        if mlflow_logging:
            print(json.loads((run_dir / "mlflow_status.json").read_text()))

## Optuna 실험

`training_mode="optuna"`일 때만 실행합니다. 위 실행 설정의 `model_name`, `model_version`, `n_trials`를 사용하며 기존 탐색 범위는 아래 objective에 있습니다. trial마다 train으로 학습하고 validation AP로 선택합니다. Trial 모델을 MLflow에 전송하지 않고, 완료한 study의 best 모델 하나만 저장·기록합니다.

개인 계정·서버·실험은 `.poc.local.toml`과 환경 변수에서 읽습니다. Class 생성 시 계정 연결을 확인하고, 완료 후 최적 모델·전처리·파라미터·validation 지표를 기록합니다. `run_final_test=True`이면 마지막 셀에서 이번 `best_run_dir`를 평가합니다.

In [ ]:
import optuna
from Members.sang.model.ML.optuna_experiment import ClinicalOptunaExperiment

experiment = None
if training_mode == "optuna":
    experiment = ClinicalOptunaExperiment(
        choice_path, model_name, model_version, mlflow_logging=mlflow_logging
    )
    parameter_path = experiment.parameter_path
    print("선택한 모델 설정:", parameter_path)
    print("최적 모델 저장 폴더:", experiment.best_directory)

## 탐색 범위와 학습 함수

아래 숫자와 목록이 모델별 탐색 범위입니다. `suggest_int`는 정수, `suggest_float`는 실수, `suggest_categorical`은 목록에서 선택합니다. `log=True`는 작은 값과 큰 값의 규모를 함께 탐색할 때 사용합니다. 자세한 범위와 역할은 `ML/ML_manu.md`의 **Optuna 확장 탐색 범위**를 참고하세요.

- XGBoost: 트리 수·깊이·학습률 + 자식 노드 조건·분할 규제·행/열 샘플링·L1/L2 규제·양성 가중치
- CatBoost: 반복 수·깊이·학습률 + L2 규제·분할 무작위성·열 샘플링·수치 분할 개수·클래스 가중치·bootstrap
- Random Forest: 트리 수 + 깊이·노드/leaf 최소 샘플·feature 샘플링·분할 기준·클래스 가중치·가지치기·bootstrap

CatBoost는 Bayesian일 때 `bagging_temperature`, Bernoulli일 때 `subsample`만 탐색합니다. Random Forest는 bootstrap이 켜져 있을 때만 `max_samples`를 탐색합니다. 이 조건을 유지해야 호환되지 않는 조합을 피할 수 있습니다. CatBoost `auto_class_weights`와 별도의 `class_weights`·`scale_pos_weight`는 함께 설정하지 마세요.

매 trial에서 **train으로 학습 → validation AP 계산**을 수행합니다. XGBoost·CatBoost의 조기 종료 횟수는 YAML 값을 사용합니다. Random Forest는 설정한 tree 수를 학습합니다. Threshold는 이번 탐색 대상이 아니며 test를 파라미터 선택에 사용하지 않습니다.

범위를 넓힌다고 성능 개선이 보장되지는 않습니다. `n_trials`는 탐색할 조합 수이므로 변수 개수가 늘면 같은 10회로는 일부 조합만 평가합니다. 더 탐색하려면 준비 셀의 `n_trials`를 늘리세요. 기본 TPE는 첫 10회에 무작위 탐색을 사용합니다. 작은 validation 결과에 맞춰 지나치게 조정하지 말고, 모델·버전·threshold를 확정한 뒤 최종 test를 평가합니다.

In [ ]:
def objective(trial):
    if model_name == "XGBoost":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1500, step=100),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0, step=0.5),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
            "reg_alpha": trial.suggest_categorical("reg_alpha", [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 30.0, log=True),
            "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1.0, 2.0, 4.0, 8.0]),
        }
    elif model_name == "CatBoost":
        params = {
            "iterations": trial.suggest_int("iterations", 100, 1500, step=100),
            "depth": trial.suggest_int("depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 30.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.0, 5.0, step=0.5),
            "rsm": trial.suggest_float("rsm", 0.6, 1.0, step=0.1),
            "border_count": trial.suggest_categorical("border_count", [16, 32, 64, 128]),
            "auto_class_weights": trial.suggest_categorical(
                "auto_class_weights", ["None", "Balanced", "SqrtBalanced"]
            ),
            "bootstrap_type": trial.suggest_categorical(
                "bootstrap_type", ["Bayesian", "Bernoulli"]
            ),
        }
        if params["bootstrap_type"] == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float(
                "bagging_temperature", 0.0, 5.0, step=0.5
            )
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.6, 1.0, step=0.1)
    else:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
            "max_depth": trial.suggest_categorical("max_depth", [None, 2, 3, 4, 5, 6, 8, 10, 15]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 12),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, None]),
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "class_weight": trial.suggest_categorical(
                "class_weight", [None, "balanced", "balanced_subsample"]
            ),
            "ccp_alpha": trial.suggest_categorical("ccp_alpha", [0.0, 0.0001, 0.001, 0.005, 0.01]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        }
        if params["bootstrap"]:
            params["max_samples"] = trial.suggest_float("max_samples", 0.6, 1.0, step=0.1)
        else:
            params["max_samples"] = None

    return experiment.run_trial(trial, params)

## 탐색 실행

validation AP가 클수록 좋은 trial을 선택합니다. 모델 학습 seed와 Optuna 제안 seed는 기본 42입니다.

**trial 모델은 임시 폴더에서 학습하고 바로 정리합니다.** 탐색 중에는 현재 best 후보 하나만 유지합니다. 탐색이 성공적으로 끝나면 해당 알고리즘·버전의 최적 모델 한 개를 `ML/models/best/<알고리즘>_v<버전>/`에 저장합니다. 예: `ML/models/best/xgboost_v4/`.

같은 조합으로 새 탐색을 완료하면 그 폴더를 새 study의 best 결과로 갱신합니다. 다른 알고리즘·버전은 유지합니다. 탐색이 실패하면 이전에 저장된 best는 보존하고 이번 임시 파일을 정리합니다. 기존에 날짜별로 저장했던 과거 폴더는 이 코드에서 삭제하지 않습니다.

이 셀을 다시 실행하려면 **Optuna 준비 셀부터 다시 실행**하여 새 experiment를 만드세요. 현재 예시는 DB 탐색 재개나 pruning 없이 순차 실행합니다.

In [ ]:
if training_mode == "optuna":
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.set_user_attr("model_name", model_name)
    study.set_user_attr("model_version", model_version)
    with experiment:
        study.optimize(objective, n_trials=n_trials)
        best_run_dir = experiment.save_best(study)

## 최적 결과 저장 및 확인

저장은 바로 앞 탐색 실행 셀에서 자동 완료됩니다. 이 셀은 `best_run_dir`의 결과를 확인합니다. 최적 모델은 이미 학습되어 있어 다시 학습할 필요가 없습니다.

한 폴더에 최적 모델 파일 한 개와 `best_parameters.yaml`을 저장합니다. 평가에 필요한 전처리·metadata·설정·validation 예측·지표·완료 상태도 함께 보관합니다. trial별 모델, 반복별 학습 이력, 반복되는 audit 파일은 영구 보관하지 않습니다.

- `trials.csv`: 이번 study의 모든 trial 파라미터·validation AP·상태만 기록
- `best_parameters.yaml`: 최적 모델의 전체 parameter 설정과 최적 모델 경로
- `summary.json`: 모델 이름·입력 버전·best AP·최적 모델 경로

예를 들어 XGBoost 버전 4를 여러 번 실행해도 `ML/models/best/xgboost_v4/` 한 곳을 갱신합니다. 탐색 완료 시 모델 YAML의 `model_path`와 버전별 `model_paths`를 자동 갱신하고, 기본 하이퍼파라미터 값은 유지합니다.

MLflow 전송 결과는 `summary.json`의 `mlflow`와 출력된 링크에서 확인합니다. 전송에 실패해도 로컬 best는 보존되며 `PENDING`으로 표시합니다. 아래 재시도 셀로 학습 없이 다시 전송할 수 있습니다. 원본 CSV·ECG 파형·환자별 예측 CSV는 이 자동 전송 대상이 아닙니다.

In [ ]:
if training_mode == "optuna":
    summary = json.loads((best_run_dir / "summary.json").read_text(encoding="utf-8"))
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("최적 모델과 파라미터 저장 폴더:", best_run_dir)
    display(pd.read_csv(best_run_dir / "trials.csv"))
    if summary.get("mlflow", {}).get("status") == "FINISHED":
        print("MLflow 결과:", summary["mlflow"]["url"])
    else:
        print("MLflow 전송 상태:", summary.get("mlflow", {"status": "DISABLED"}))

## MLflow 전송 재시도 (필요할 때만)

기본 `retry_mlflow=False`로 전체 실행 시 재전송을 반복하지 않습니다. 전송에 실패한 run을 재전송하려면 `retry_mlflow=True`로 바꾸고 이 셀만 실행합니다. 학습 설정 셀을 다시 실행하면 기존 run 변수도 초기화되므로 재학습 없이 재전송할 때는 아래 셀에서 직접 `retry_mlflow=True`를 지정해 실행하세요. 일반 학습과 Optuna best 모두 지원합니다.

In [ ]:
if retry_mlflow:
    from Members.sang.model.tracking.log_best import log_ml_training_to_mlflow

    selected_runs = [best_run_dir] if training_mode == "optuna" else run_dirs
    for selected_run in selected_runs:
        result = log_ml_training_to_mlflow(selected_run)
        print("MLflow 재전송 결과:", result["url"])

## 최종 test 평가와 MLflow 기록

`run_final_test=True`이면 같은 노트북에서 이번에 선택한 모델을 공통 evaluation으로 평가합니다. YAML 학습은 `run_dirs` 각각, Optuna는 `best_run_dir` 하나를 사용합니다. 저장된 best 모델·전처리·threshold를 그대로 적용하며 평가 단계에서는 학습하지 않습니다.

`mlflow_logging=True`이면 학습 때와 같은 개인 계정으로 별도 평가 run을 기록하고 원래 학습 run을 연결합니다. `eval.yaml` 원본은 수정하지 않습니다. 후보 비교·튜닝 중에는 `run_final_test=False`로 설정하고, 최종 설정을 고정한 뒤 실행하세요.

In [ ]:
if run_final_test:
    from tempfile import TemporaryDirectory

    import yaml
    from Members.sang.model.config import load_experiment_config, load_yaml, resolve_config_path
    from Members.sang.model.evaluation.evaluation import evaluate_saved_model

    _, _, _, basic_path = load_experiment_config(choice_path)
    eval_path = repo / "Members/sang/model/evaluation/eval.yaml"
    config = load_yaml(eval_path)
    config["basic_config"] = str(basic_path)
    config["evaluation"].update(
        target="ml",
        split="test",
        threshold_override=None,
        mlflow_logging=mlflow_logging,
        results_directory=str(
            resolve_config_path(config["evaluation"]["results_directory"], eval_path)
        ),
    )
    selected_runs = [best_run_dir] if training_mode == "optuna" else run_dirs
    with TemporaryDirectory(prefix="clinical-evaluation-") as temporary:
        temporary_config = Path(temporary) / "eval.yaml"
        for selected_run in selected_runs:
            metadata = json.loads((selected_run / "metadata.json").read_text())
            config["evaluation"].update(
                artifact_dir=str(selected_run),
                model_name=metadata["model"],
                model_version=metadata["model_version"],
            )
            temporary_config.write_text(yaml.safe_dump(config), encoding="utf-8")
            evaluation_dir = evaluate_saved_model(temporary_config)
            evaluation_dirs.append(evaluation_dir)
            print("평가 결과:", evaluation_dir)
            print(json.loads((evaluation_dir / "metrics.json").read_text()))
            display(pd.read_csv(evaluation_dir / "predictions.csv").head())
            if mlflow_logging:
                print(json.loads((evaluation_dir / "mlflow_status.json").read_text()))